Classificatore specie:
- input: immagine
- output label: nativa/non nativa e non invasiva/non nativa e invasiva
La struttura del modello utilizzato è simile a quella precedentemente implementata:
le immagini passano attraverso un modello preallenato (BioCLIP o ResNet) da cui vengono estratti gli embedding
gli embedding allenano un semplice classificatore.

La loss function è stata modificata: ogni classe ha il suo peso (dipendente dalla frequenza della classe).

Durante il training le fasi di estrazione degli embedding e predizione del label sono separate (così che il training sia molto più veloce e leggero). Una volta finito il training viene ricostruito il modello intero, così che il classificatore prenda come input l'immagine (non l'embedding) e restituisca il label (in modo che si possa effettuare successivamente explainability sulle immagini e non solo sugli embedding).

Rispetto al modello di Barbara, estraggo gli embeddings tutti insieme (e non separati per training set e validation set), così da poter effettuare cross validation senza dover ogni volta estrarre gli embeddings.

In [1]:
#@title Imports and downloads
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import functional as TF
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
from torchvision import transforms, models
from sklearn.metrics import classification_report, multilabel_confusion_matrix, confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import GroupKFold
from google.colab import drive
import time
from tqdm import tqdm
!pip install open_clip_torch
import open_clip
import math
from collections import Counter
import datetime
import json
import os

torch.manual_seed(42)
np.random.seed(42)

drive.mount('/content/drive')

# Clone the repository and checkout the 'clean-barbara' branch
!git clone --branch clean-barbara https://github.com/babi00/ai4biological-pattern.git
%cd ai4biological-pattern

# Enable sparse checkout
!git sparse-checkout init --cone

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path 'ai4biological-pattern' already exists and is not an empty directory.
/content/ai4biological-pattern


In [2]:
#@title Training log saving function
def save_training_log(model_name, output_dir, train_losses, val_losses, train_accuracies, val_accuracies, classification_report_dict):
    import datetime
    import json
    import os

    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_dir = f"/content/drive/MyDrive/Thesis/{output_dir}/logs"
    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, "training_log.json")

    # Load existing logs (if any)
    if os.path.exists(log_path):
        with open(log_path, "r") as f:
            all_logs = json.load(f)
    else:
        all_logs = []

    # Prepare this run's data
    this_log = {
        "model_name": model_name,
        "timestamp": timestamp,
        "final_train_loss": train_losses[-1] if train_losses else None,
        "final_val_loss": val_losses[-1] if val_losses else None,
        "final_train_accuracy": train_accuracies[-1] if train_accuracies else None,
        "final_val_accuracy": val_accuracies[-1] if val_accuracies else None,
        "classification_report_last_epoch": classification_report_dict[-1] if isinstance(classification_report_dict, list) else classification_report_dict,
        "train_loss_history": train_losses,
        "val_loss_history": val_losses,
        "train_accuracy_history": train_accuracies,
        "val_accuracy_history": val_accuracies
    }

    all_logs.append(this_log)

    # Save updated list
    with open(log_path, "w") as f:
        json.dump(all_logs, f, indent=4)

    print(f"📝 Appended log to {log_path}")


In [3]:
#@title Horizontal flip
def hflip(image):
  return TF.hflip(image)

In [4]:
#@title Original Dataset
class InvasiveSpeciesDataset(Dataset):
    def __init__(self, root_dir, transform=None, get_label_fn=None):
        self.root_dir = root_dir
        self.transform = transform
        self.entries = []
        self.label_map = {
            "native": 0,
            "introduced non-invasive": 1,
            "invasive": 2
        }

        taxa_folders = sorted(
            [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d)) and d != "metadata"]
        )

        for taxon in taxa_folders:
            meta_path = os.path.join(root_dir, "filtered_metadata", f"{taxon}_metadata.csv")
            if not os.path.exists(meta_path):
                continue
            metadata_df = pd.read_csv(meta_path)
            for idx, row in metadata_df.iterrows():
                self.entries.append((taxon, row))

        # # ✅ Sort all entries by taxon name and filename to make everything stable
        # self.entries.sort(key=lambda x: (x[0], x[1]["filename"]))


    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        taxon, row = self.entries[idx]
        filename = str(row["filename"])
        image_path = os.path.join(self.root_dir, taxon, filename)
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.label_map.get(row["label"])
        return image, label, idx

In [5]:
#@title Augmented Dataset
class AugmentedClass1Dataset(Dataset):
  def __init__(self, original_dataset, transform=None):
    self.original_dataset = original_dataset
    self.transform = transform
    self.entries = [entry for entry in original_dataset.entries if original_dataset.label_map.get(entry[1]["label"]) == 1]
    self.root_dir = original_dataset.root_dir

  def __len__(self):
    return len(self.entries)

  def __getitem__(self, idx):
    taxon, row = self.entries[idx]
    filename = str(row['filename'])
    image_path = os.path.join(self.root_dir, taxon, filename)
    image = Image.open(image_path).convert("RGB")
    if self.transform:
        image = self.transform(image)
    label = 1
    return image, label, idx

In [6]:
#@title Return the dataloader of all images and not already separated (also class weights are not present)
def get_data_loader(root_dir, batch_size=32, use_bioclip=True, preprocess=None):
    if use_bioclip and preprocess:
        transform = preprocess
    else:
        transform = transforms.Compose([
          transforms.Resize((224, 224)),
          transforms.ToTensor()
        ])

    dataset = InvasiveSpeciesDataset(
        root_dir=root_dir,
        transform=transform,
    )

    loader = DataLoader(dataset, batch_size=32, shuffle=False)

    return loader

In [7]:
#@title Augmentation helper functions
def get_augmented_transform_flipping(preprocess=None):

    if preprocess:
        # BioCLIP: use its own preprocessing, plus flipping
        return transforms.Compose([
            transforms.Lambda(lambda img: TF.hflip(img)),
            preprocess
        ])
    else:
        # ResNet or default: manual transform
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.Lambda(lambda img: TF.hflip(img)),
            transforms.ToTensor()
        ])


def get_augmented_transform_jitter(preprocess=None):

    if preprocess:
        # BioCLIP: use its own preprocessing, plus jittering (no hue modification)
        return transforms.Compose([
            transforms.ColorJitter(0.3, 0.3, 0.3),
            preprocess
        ])
    else:
        # ResNet or default: manual transform
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ColorJitter(0.3, 0.3, 0.3),
            transforms.ToTensor()
        ])

In [8]:
#@title Augmented DataLoader
def get_data_loader_augmented(root_dir, batch_size=32, use_bioclip=True, preprocess=None, augmentation='flipping'):
    if use_bioclip and preprocess:
        original_transform = preprocess
    else:
        original_transform = transforms.Compose([
          transforms.Resize((224, 224)),
          transforms.ToTensor(),
          # transforms.RandomHorizontalFlip(p=1.0)
        ])

    if augmentation=='flipping':
      augmented_transform = get_augmented_transform_flipping(preprocess if use_bioclip else None)
    elif augmentation=='jittering':
      augmented_transform = get_augmented_transform_jitter(preprocess if use_bioclip else None)
    else:
      print("Error in the augmentation!")
      augmented_transform = -1 #this will break after


    original_dataset = InvasiveSpeciesDataset(
        root_dir=root_dir,
        transform=original_transform,
    )

    augmented_dataset = AugmentedClass1Dataset(
        original_dataset=original_dataset,
        transform=augmented_transform,
    )

    augmented_loader = DataLoader(augmented_dataset, batch_size=32, shuffle=False)

    return augmented_loader

In [9]:
#@title 2.Define the embedding extractor:
#ResNet without the last FC layer
def get_resnet_embeddings_extractor_model():
    #Use a pre-trained ResNet18 model
    model = models.resnet18(weights='IMAGENET1K_V1')

    #Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    embeddings_extractor= nn.Sequential(*list(model.children())[:-1]) #removes the last FC layer
    print("Model obtained...")

    return embeddings_extractor

#BioCLIP model
def get_bioclip_embeddings_extractor_model(device, bioclip_version=2):
    if bioclip_version==2:
        print("BioCLIP 2!")
        model, _, preprocess = open_clip.create_model_and_transforms('hf-hub:imageomics/bioclip-2')
    else:
        model, _, preprocess = open_clip.create_model_and_transforms('hf-hub:imageomics/bioclip')
    print("Model obtained...")
    model.to(device)
    model.eval()
    return model, preprocess

In [10]:
#@title 3. Extract embeddings using the embeddings_extractor and return the embeddings tensors
def extract_embeddings(dataloader, embeddings_extractor, device, embeddings_save_path, augmented=False, use_bioclip=True, dataset=None):
    embeddings_extractor.eval()
    all_embeddings = []
    all_labels = []
    all_ids = []
    all_groups = []  # ✅ NEW: store taxa names

    with torch.no_grad():
      for images, labels, obs_id in tqdm(dataloader, desc='Extracting embeddings'):
          images = images.to(device)
          if use_bioclip:
              embeddings = embeddings_extractor.encode_image(images)
              embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)  # Optional: normalize
          else:
              embeddings = embeddings_extractor(images).view(images.size(0), -1)

          all_embeddings.append(embeddings.cpu())
          all_labels.append(labels)
          all_ids.extend(obs_id)

          # ✅ NEW: reconstruct taxa names directly here
          if dataset is not None:
              for i in obs_id:
                  taxon, _ = dataset.entries[int(i)]
                  all_groups.append(taxon)

    embeddings_tensors = torch.cat(all_embeddings, dim=0).to(device)
    labels_tensors = torch.cat(all_labels, dim=0).to(device)
    obs_id_tensor = torch.tensor(all_ids).to(device)

    all_embeddings = {
       'embeddings' : embeddings_tensors,
       'labels' : labels_tensors,
       'ids' : obs_id_tensor,
       'groups': all_groups  # ✅ Save group names
    }

    if augmented==False:
      torch.save(all_embeddings, embeddings_save_path)

    # embedding_size = embeddings_tensors.shape[1] #useful for classifier

    return all_embeddings

In [11]:
#@title Split the embeddings 80-20, then add the augmentation to the training set
def split_embeddings(embeddings_dict, augmented_embeddings_dict=None, val_percentage=0.2):

    """Split the embeddings and not the images into training set and validation set"""

    embeddings_tensors = embeddings_dict["embeddings"]
    labels_tensors = embeddings_dict["labels"]
    obs_id_tensor = embeddings_dict["ids"]

    embedding_size = embeddings_tensors.shape[1] #useful for classifier

    dataset = torch.utils.data.TensorDataset(embeddings_tensors, labels_tensors, obs_id_tensor)

    train_size = int((1-val_percentage) * len(dataset))
    val_size = len(dataset) - train_size

    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    #if using augmentation
    if augmented_embeddings_dict:
      print("Augmenting train dataset...")
      aug_embeddings_tensors = augmented_embeddings_dict["embeddings"]
      aug_labels_tensors = augmented_embeddings_dict["labels"]
      aug_obs_id_tensor = augmented_embeddings_dict["ids"]

      augmented_train_dataset = torch.utils.data.TensorDataset(aug_embeddings_tensors, aug_labels_tensors, aug_obs_id_tensor)

      #concatenate train dataset with augmented train dataset
      train_dataset = ConcatDataset([train_dataset, augmented_train_dataset])

    label_counts = Counter()
    for _, label, _ in train_dataset:
        label_counts[int(label)] += 1

    print("🔢 Class distribution in train set:", label_counts)

    # total = sum(label_counts.values())
    class_weights = torch.tensor([
        1.0 / math.log(1.02 + label_counts[0]),
        1.0 / math.log(1.02 + label_counts[1]),
        1.0 / math.log(1.02 + label_counts[2]),
    ], dtype=torch.float)

    print("⚖️ Class weights:", class_weights)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=True)

    return train_loader, val_loader, class_weights, embedding_size



In [12]:
#@title K fold helper function

def inspect_folds_groups(dataset, n_splits=5):
    """
    Print the number of validation groups, their names,
    and the class distribution for each fold (GroupKFold).
    """
    labels = [dataset.label_map[row["label"]] for _, row in dataset.entries]
    groups = [taxon for taxon, _ in dataset.entries]

    gkf = GroupKFold(n_splits=n_splits)

    print(f"🔍 Inspecting {n_splits}-fold GroupKFold:")
    for fold_idx, (_, val_idx) in enumerate(gkf.split(np.zeros(len(labels)), labels, groups)):
        val_groups = sorted(set(groups[i] for i in val_idx))
        val_labels = [labels[i] for i in val_idx]
        class_counts = Counter(val_labels)

        print(f"\n📂 Fold {fold_idx}: {len(val_groups)} validation groups")
        print(f"Class distribution: {dict(class_counts)}")
        print(f"Species in this fold: {val_groups}")

def inspect_folds_groups_with_augmentation(embeddings_dict, augmented_embeddings_dict=None, dataset=None, n_splits=5):
    """
    Print the number of validation groups, their names,
    and the class distribution for each fold using the ACTUAL embeddings data,
    INCLUDING augmentation effects on training sets.
    """
    labels_tensors = embeddings_dict["labels"]
    obs_id_tensor = embeddings_dict["ids"]

    # Reconstruct groups from the actual embeddings data
    labels = labels_tensors.cpu().numpy()
    groups = []
    for idx in obs_id_tensor:
        taxon, _ = dataset.entries[int(idx)]
        groups.append(taxon)

    groups = np.array(groups)

    gkf = GroupKFold(n_splits=n_splits)

    print(f"🔍 Inspecting {n_splits}-fold GroupKFold with augmentation effects:")
    print(f"📊 Total samples in original embeddings: {len(labels)}")
    print(f"📊 Total unique groups: {len(set(groups))}")

    if augmented_embeddings_dict:
        aug_labels = augmented_embeddings_dict["labels"].cpu().numpy()
        print(f"📊 Total augmented samples (class 1 only): {len(aug_labels)}")
        print(f"📊 Augmented class distribution: {dict(Counter(aug_labels))}")

    for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(np.zeros(len(labels)), labels, groups)):
        # Validation set (unchanged)
        val_groups = sorted(set(groups[i] for i in val_idx))
        val_labels = [labels[i] for i in val_idx]
        val_class_counts = Counter(val_labels)

        # Training set (original)
        train_labels = [labels[i] for i in train_idx]
        train_class_counts = Counter(train_labels)

        print(f"\n📂 Fold {fold_idx}: {len(val_groups)} validation groups")
        print(f"VALIDATION - Class distribution: {dict(val_class_counts)}")
        print(f"VALIDATION - Total samples: {len(val_labels)}")
        print(f"TRAINING (original) - Class distribution: {dict(train_class_counts)}")
        print(f"TRAINING (original) - Total samples: {len(train_labels)}")

        # Add augmentation effects to training set
        if augmented_embeddings_dict:
            # All augmented samples go to training (they're all class 1)
            aug_labels = augmented_embeddings_dict["labels"].cpu().numpy()
            combined_train_labels = list(train_labels) + list(aug_labels)
            combined_train_class_counts = Counter(combined_train_labels)

            print(f"TRAINING (with augmentation) - Class distribution: {dict(combined_train_class_counts)}")
            print(f"TRAINING (with augmentation) - Total samples: {len(combined_train_labels)}")

        print(f"Species in validation fold: {val_groups}")

def debug_validation_set_actual_with_augmentation(embeddings_dict, augmented_embeddings_dict=None, dataset=None, n_splits=5, fold_idx=0):
    """
    Debug function to print what's actually in the training and validation sets for a specific fold,
    including augmentation effects.
    """
    labels_tensors = embeddings_dict["labels"]
    obs_id_tensor = embeddings_dict["ids"]

    labels = labels_tensors.cpu().numpy()
    groups = []
    for idx in obs_id_tensor:
        taxon, _ = dataset.entries[int(idx)]
        groups.append(taxon)

    groups = np.array(groups)

    gkf = GroupKFold(n_splits=n_splits)
    all_splits = list(gkf.split(np.zeros(len(labels)), labels, groups))
    train_idx, val_idx = all_splits[fold_idx]

    # Validation set (unchanged)
    val_labels = labels[val_idx]
    val_groups = groups[val_idx]

    # Training set (original)
    train_labels = labels[train_idx]
    train_groups = groups[train_idx]

    print(f"\n🔍 DEBUG: Actual sets for fold {fold_idx}:")
    print(f"VALIDATION SET:")
    print(f"  Size: {len(val_labels)}")
    print(f"  Class distribution: {dict(Counter(val_labels))}")
    print(f"  Unique groups: {sorted(set(val_groups))}")

    print(f"TRAINING SET (original):")
    print(f"  Size: {len(train_labels)}")
    print(f"  Class distribution: {dict(Counter(train_labels))}")
    print(f"  Unique groups: {sorted(set(train_groups))}")

    # Add augmentation effects
    if augmented_embeddings_dict:
        aug_labels = augmented_embeddings_dict["labels"].cpu().numpy()
        combined_train_labels = list(train_labels) + list(aug_labels)

        print(f"TRAINING SET (with augmentation):")
        print(f"  Size: {len(combined_train_labels)}")
        print(f"  Class distribution: {dict(Counter(combined_train_labels))}")
        print(f"  Added {len(aug_labels)} augmented samples (all class 1)")

        # Show the difference augmentation makes
        original_class_1 = Counter(train_labels)[1]
        augmented_class_1 = Counter(combined_train_labels)[1]
        print(f"  Class 1 samples: {original_class_1} → {augmented_class_1} (+{augmented_class_1 - original_class_1})")

    return val_labels, val_groups, train_labels

In [13]:
#@title Group K Fold

from sklearn.model_selection import GroupKFold

def split_embeddings_groupkfold(embeddings_dict, augmented_embeddings_dict=None, n_splits=5, fold_idx=0):
    """
    Split the embeddings using GroupKFold instead of random_split.
    Automatically uses taxon as groups (reconstructed from dataset.entries).
    """

    embeddings_tensors = embeddings_dict["embeddings"]
    labels_tensors = embeddings_dict["labels"]
    obs_id_tensor = embeddings_dict["ids"]

    embedding_size = embeddings_tensors.shape[1]  # useful for classifier

    groups_np = np.array(embeddings_dict["groups"])  # ✅ Directly use stored groups

    labels_np = labels_tensors.cpu().numpy()

    gkf = GroupKFold(n_splits=n_splits)
    all_splits = list(gkf.split(np.zeros(len(labels_np)), labels_np, groups_np))
    train_idx, val_idx = all_splits[fold_idx]

    #build datasets
    train_dataset = torch.utils.data.TensorDataset(
        embeddings_tensors[train_idx],
        labels_tensors[train_idx],
        obs_id_tensor[train_idx]
    )
    val_dataset = torch.utils.data.TensorDataset(
        embeddings_tensors[val_idx],
        labels_tensors[val_idx],
        obs_id_tensor[val_idx]
    )

    #augmentation
    if augmented_embeddings_dict:
        print("Augmenting train dataset...")
        aug_embeddings_tensors = augmented_embeddings_dict["embeddings"]
        aug_labels_tensors = augmented_embeddings_dict["labels"]
        aug_obs_id_tensor = augmented_embeddings_dict["ids"]
        augmented_train_dataset = torch.utils.data.TensorDataset(
            aug_embeddings_tensors, aug_labels_tensors, aug_obs_id_tensor
        )
        train_dataset = ConcatDataset([train_dataset, augmented_train_dataset])

    #class weights (as before)
    label_counts = Counter()
    for _, label, _ in train_dataset:
        label_counts[int(label)] += 1

    print("🔢 Class distribution in train set:", label_counts)
    class_weights = torch.tensor([
        1.0 / math.log(1.02 + label_counts.get(0, 1)),
        1.0 / math.log(1.02 + label_counts.get(1, 1)),
        1.0 / math.log(1.02 + label_counts.get(2, 1)),
    ], dtype=torch.float)
    print("⚖️ Class weights:", class_weights)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)

    return train_loader, val_loader, class_weights, embedding_size


In [14]:
#@title Classifier
class InvasiveSpeciesClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim=256, num_classes=3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(), #consider adding a Dropout layer
            # nn.Dropout(0.3), #We remove the dropout layer
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)

In [15]:
#@title Early Stopping class

#Patience: how many epochs to wait after last improvements
#delta: minimum change in validation to be considered improvement
class EarlyStopping:

    def __init__(self, patience=5, delta=0.01, verbose=False):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.best_loss = None
        self.no_improvement_count = 0
        self.stop_training = False

    def check_early_stop(self, val_loss):
        if self.best_loss is None or val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.no_improvement_count = 0
        else:
            self.no_improvement_count += 1
            if self.no_improvement_count >= self.patience:
                self.stop_training = True
                if self.verbose:
                    print("Stopping early as no improvement has been observed.")

In [16]:
#@title 4. Training function
def train_classifier(train_loader, val_loader, feature_extractor, classifier, optimizer, criterion, device, output_dir, model_name, num_epoch=10, verbose=True, save_classifier=False):

    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    early_stopping = EarlyStopping(patience=50, delta=0, verbose=True)

    for epoch in range(num_epoch):
        print(f"\n🔁 Epoch {epoch + 1}/{num_epoch}")
        classifier.train()
        running_loss, correct, total = 0, 0, 0

        for embeddings, labels, _ in tqdm(train_loader, desc="Training"): # Unpack all three values
            outputs = classifier(embeddings)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * embeddings.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        val_loss, val_acc, y_true, y_pred = evaluate_classifier(
            val_loader, feature_extractor, classifier, criterion, device
        )

        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        if verbose:
            print(f"📉 Train Loss: {train_loss:.4f} | Accuracy: {train_acc:.4f}")
            print(f"📈 Val Loss:   {val_loss:.4f} | Accuracy: {val_acc:.4f}")

            print("\n📋 Classification Report:")
            report_dict = classification_report(y_true, y_pred, labels=[0, 1, 2], target_names=["Native", "NN Non-Invasive", "NN Invasive"], zero_division=0, output_dict=True)
            print(classification_report(y_true, y_pred, labels=[0, 1, 2], target_names=["Native", "NN Non-Invasive", "NN Invasive"], zero_division=0, output_dict=False))

            #plot confusion matrix
            cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
            fig, ax = plt.subplots(figsize=(6, 6))
            im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
            ax.figure.colorbar(im, ax=ax)
            classes = ["Native", "NN-NI", "NN-I"]

            # Show all ticks and label them
            ax.set(xticks=np.arange(len(classes)),
                yticks=np.arange(len(classes)),
                xticklabels=classes,
                yticklabels=classes,
                ylabel='True label',
                xlabel='Predicted label',
                title='Validation Confusion Matrix')

            # Rotate the tick labels and set alignment.
            plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

            # Loop over data dimensions and create text annotations.
            fmt = 'd'
            thresh = cm.max() / 2. if cm.max() > 0 else 1
            for i in range(cm.shape[0]):
                for j in range(cm.shape[1]):
                    ax.text(j, i, format(cm[i, j], fmt),
                            ha="center", va="center",
                            color="white" if cm[i, j] > thresh else "black")

            fig.tight_layout()
            plt.grid(False)
            plt.show()
        else:
          report_dict = None

        # Check early stopping condition
        early_stopping.check_early_stop(val_loss)

        if early_stopping.stop_training:
          print(f"Early stopping at epoch {epoch}")
          break


    torch.save(classifier.state_dict(), f'classifier_head_{model_name}.pth')
    google_drive_path = f'/content/drive/MyDrive/Thesis/{output_dir}/classifier_head_{model_name}.pth'
    torch.save(classifier.state_dict(), google_drive_path)

    return train_losses, train_accuracies, val_losses, val_accuracies, report_dict

In [17]:
#@title 5. Evaluation function

from sklearn.metrics import balanced_accuracy_score

def evaluate_classifier(loader, feature_extractor, classifier, criterion, device):
    classifier.eval()
    total_loss, correct, total = 0, 0, 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for embeddings, labels, obs_ids in tqdm(loader, desc="Evaluating"): # Unpack all three values
            embeddings, labels = embeddings.to(device), labels.to(device)
            outputs = classifier(embeddings)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * embeddings.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / total
    avg_acc = correct / total

    #I AM RETURNING THE BALANCED ACCURACY
    # balanced_acc = balanced_accuracy_score(all_labels, all_preds)

    return avg_loss, avg_acc, all_labels, all_preds

In [18]:
#@title Focal loss function

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        """
        alpha: Class weighting (tensor or list), same as in CrossEntropyLoss
        gamma: Focusing parameter (higher → more focus on hard samples) (gamma=0 => cross entropy loss)
        reduction: 'mean', 'sum', or 'none'
        """
        super(FocalLoss, self).__init__()
        self.alpha=alpha
        self.gamma=gamma
        self.reduction=reduction

    def forward(self, inputs, targets):
        #cross entropy per sample (no reduction)
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')

        #convert to probabilities of true class for modulating factor
        pt = torch.exp(-ce_loss) #ce = -log(pt), so pt = e^(-ce), model confidence of true class

        #focal loss formula (focal term)
        focal_loss = ((1-pt)**self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean() #return average loss per batch
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss


In [19]:
#@title Full Model
class FullModel(nn.Module):
    def __init__(self, embedding_model, classifier_head):
        super().__init__()
        self.embedding_model = embedding_model
        self.classifier_head = classifier_head

    def forward(self, x):
        with torch.no_grad():
            if hasattr(self.encoder, 'encode_image'):
                features = self.encoder.encode_image(x)
            else:
                features = self.encoder(x)

        if features.ndim == 4:
            features = features.view(features.size(0), -1)

        return self.classifier_head(features)

In [20]:
#@title Create model and predict
def create_model_and_predict(folder_name="taxas", use_bioclip=True, bioclip_version=None,
                             augmentation=True, augmentation_factor=2, loss_type='CE', output_dir="invasive_species",
                             model_name="prova", verbose=True, save_model=True, embeddings_save_path=None,
                             epochs=50, debug_subset_size=None, n_splits=5, fold_idx=0):
    start_time = time.time()

    model_name = f'bioclip_2_{model_name}' if use_bioclip and bioclip_version==2 else f'bioclip_1_{model_name}' if use_bioclip else f'resnet_{model_name}'

    github = '/content/ai4biological-pattern/barbara_new'
    github_repository = f'{github}/{folder_name}'

    if not os.path.isdir(github_repository): #check if data has already been downloaded
        !git sparse-checkout set barbara_new/{folder_name}
        !git checkout clean-barbara

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    if use_bioclip:
        feature_extractor, preprocess = get_bioclip_embeddings_extractor_model(device, bioclip_version)
        feature_extractor = feature_extractor.to(device)
    else:
        feature_extractor = get_resnet_embeddings_extractor_model()
        feature_extractor = feature_extractor.to(device)
        preprocess = None
    print("Feature extractor obtained..")

    # def count_unique_taxa(dataset):
    #   groups = [taxon for taxon, _ in dataset.entries]
    #   unique_groups = set(groups)
    #   print(f"📊 Total number of taxa (groups): {len(unique_groups)}")
    #   return unique_groups

    #Load dataset to reconstruct groups
    dataset = InvasiveSpeciesDataset(root_dir=github_repository, transform=preprocess if use_bioclip else None)
    # inspect_folds_groups(dataset, n_splits=n_splits)
    # count_unique_taxa(dataset)

    if os.path.exists(embeddings_save_path):

        print("✅ Embeddings already exist, loading from file...")
        embeddings_dict = torch.load(embeddings_save_path)

        #now get only images of class 1 from the original dataset, pass them to the function and concatenate the original training dataset and the augmented dataset
        img_loader = get_data_loader(root_dir=github_repository)

        if augmentation:
          #only extract embeddings for augmentated data
          augmented_loader = get_data_loader_augmented(root_dir=github_repository, augmentation='flipping')
          augmented_embeddings_dict = extract_embeddings(augmented_loader, feature_extractor, device, embeddings_save_path=embeddings_save_path, augmented=True, use_bioclip=use_bioclip, dataset=dataset)

          if augmentation_factor==3:
            augmented_loader_2 = get_data_loader_augmented(root_dir=github_repository, augmentation='jittering')
            augmented_embeddings_dict_2 = extract_embeddings(augmented_loader_2, feature_extractor, device, embeddings_save_path=embeddings_save_path, augmented=True, use_bioclip=use_bioclip, dataset=dataset)

            augmented_embeddings_dict = {
              "embeddings": torch.cat([augmented_embeddings_dict["embeddings"], augmented_embeddings_dict_2["embeddings"]]),
              "labels": torch.cat([augmented_embeddings_dict["labels"], augmented_embeddings_dict_2["labels"]]),
              "ids": torch.cat([augmented_embeddings_dict["ids"], augmented_embeddings_dict_2["ids"]])
            }

        else:
          augmented_embeddings_dict=None

        inspect_folds_groups_with_augmentation(embeddings_dict, augmented_embeddings_dict, dataset, n_splits=n_splits)

        # Debug the specific fold you're training on
        debug_validation_set_actual_with_augmentation(embeddings_dict, augmented_embeddings_dict, dataset,
                                                    n_splits=n_splits, fold_idx=fold_idx)

        if debug_subset_size is not None:
          print(f"⚠ Debug mode: using only the first {debug_subset_size} embeddings.")
          embeddings_dict['embeddings'] = embeddings_dict['embeddings'][:debug_subset_size]
          embeddings_dict['labels'] = embeddings_dict['labels'][:debug_subset_size]
          embeddings_dict['ids'] = embeddings_dict['ids'][:debug_subset_size]

          if augmentation:
            augmented_embeddings_dict['embeddings'] = augmented_embeddings_dict['embeddings'][:debug_subset_size]
            augmented_embeddings_dict['labels'] = augmented_embeddings_dict['labels'][:debug_subset_size]
            augmented_embeddings_dict['ids'] = augmented_embeddings_dict['ids'][:debug_subset_size]
          else:
            augmented_embeddings_dict=None


        embeddings_train_loader, embeddings_val_loader, class_weights, input_size = split_embeddings_groupkfold(
               embeddings_dict, augmented_embeddings_dict, n_splits=n_splits, fold_idx=fold_idx
        )

    else:

        img_loader = get_data_loader(root_dir=github_repository)
        print("Dataloader obtained..")

        #this function also saves embeddings in the desired path
        #input size is the same for training and validation, is is the size (number of features) of the single embedding.
        embeddings_dict = extract_embeddings(img_loader, feature_extractor, device, use_bioclip=use_bioclip, embeddings_save_path=embeddings_save_path, dataset=dataset)

        if augmentation:
          #only extract embeddings for augmentated data
          augmented_loader = get_data_loader_augmented(root_dir=github_repository)
          augmented_embeddings_dict = extract_embeddings(augmented_loader, feature_extractor, device, embeddings_save_path=embeddings_save_path, augmented=True, use_bioclip=use_bioclip, dataset=dataset)

          if augmentation_factor==3:
            augmented_loader_2 = get_data_loader_augmented(root_dir=github_repository, augmentation='jittering')
            augmented_embeddings_dict_2 = extract_embeddings(augmented_loader_2, feature_extractor, device, embeddings_save_path=embeddings_save_path, augmented=True, use_bioclip=use_bioclip, dataset=dataset)

            augmented_embeddings_dict = {
              "embeddings": torch.cat([augmented_embeddings_dict["embeddings"], augmented_embeddings_dict_2["embeddings"]]),
              "labels": torch.cat([augmented_embeddings_dict["labels"], augmented_embeddings_dict_2["labels"]]),
              "ids": torch.cat([augmented_embeddings_dict["ids"], augmented_embeddings_dict_2["ids"]])
            }

        else:
          augmented_embeddings_dict=None

        if debug_subset_size is not None:
          print(f"⚠ Debug mode: using only the first {debug_subset_size} embeddings.")
          embeddings_dict['embeddings'] = embeddings_dict['embeddings'][:debug_subset_size]
          embeddings_dict['labels'] = embeddings_dict['labels'][:debug_subset_size]
          embeddings_dict['ids'] = embeddings_dict['ids'][:debug_subset_size]

          if augmentation:
            augmented_embeddings_dict['embeddings'] = augmented_embeddings_dict['embeddings'][:debug_subset_size]
            augmented_embeddings_dict['labels'] = augmented_embeddings_dict['labels'][:debug_subset_size]
            augmented_embeddings_dict['ids'] = augmented_embeddings_dict['ids'][:debug_subset_size]
          else:
            augmented_embeddings_dict=None

        embeddings_train_loader, embeddings_val_loader, class_weights, input_size = split_embeddings_groupkfold(
               embeddings_dict, augmented_embeddings_dict, n_splits=n_splits, fold_idx=fold_idx
        )


    classifier = InvasiveSpeciesClassifier(embedding_dim=input_size).to(device)
    if loss_type=='CE':
      criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    elif loss_type=='FL':
      criterion = FocalLoss(alpha=class_weights.to(device), gamma=2, reduction='mean')
    optimizer = optim.Adam(classifier.parameters(), lr=1e-4)


    train_losses, train_accuracies, val_losses, val_accuracies, report_dict = train_classifier(
        train_loader=embeddings_train_loader,
        val_loader=embeddings_val_loader,
        feature_extractor=feature_extractor,
        classifier=classifier,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        output_dir=output_dir,
        model_name=model_name,
        num_epoch=epochs,
        verbose=verbose,
        save_classifier=save_model
    )

    # Save results
    save_training_log(model_name, output_dir, train_losses, val_losses, train_accuracies, val_accuracies, report_dict)

    epochs = range(1, len(train_losses) + 1)

    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, 'bo-', label='Train Loss')
    plt.plot(epochs, val_losses, 'ro-', label='Val Loss')
    plt.title("Loss over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_accuracies, 'bo-', label='Train Accuracy')
    plt.plot(epochs, val_accuracies, 'ro-', label='Val Accuracy')
    plt.title("Accuracy over Epochs")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

    full_model = FullModel(feature_extractor, classifier)
    print("Full model built...")
    full_model.eval()

    if save_model:
      torch.save(full_model.state_dict(), f'full_{model_name}.pth')
      google_drive_path = f'/content/drive/MyDrive/Thesis/{output_dir}/full_{model_name}.pth'
      torch.save(full_model.state_dict(), google_drive_path)

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Total execution time: {elapsed_time:.2f} seconds")

    #return the validation loss and accuracy for the last epoch
    return val_losses[-1], val_accuracies[-1]

## Running the models

In [ ]:
all_val_losses = []
all_val_accuracies = []

n_splits = 5

final_HP_search = {'augmentation': True, 'loss_type': 'CE', 'factor': 2, 'val_loss': 0.35223526106626696, 'val_accuracy': 0.8417397244677217}
embeddings_save_path_BC2_new = '/content/drive/MyDrive/Thesis/NEW_GROUPS_bioclip_2_embeddings.pt'
model_name_final = 'group_k_fold_new_embeddings'
epochs=50

for fold_idx in range(n_splits):
  val_loss, val_acc = create_model_and_predict(folder_name="taxas", use_bioclip=True, augmentation=final_HP_search['augmentation'],
                                          augmentation_factor=final_HP_search['factor'], loss_type=final_HP_search['loss_type'],
                                         bioclip_version=2, output_dir="invasive_species", model_name=model_name_final,
                                           save_model=False, verbose=True, embeddings_save_path=embeddings_save_path_BC2_new,
                                             epochs=epochs, debug_subset_size=None, n_splits=n_splits, fold_idx=fold_idx)
  all_val_losses.append(val_loss)
  all_val_accuracies.append(val_acc)
  print(f"✅ Fold {fold_idx} done: Val Loss = {val_loss:.4f}, Val Acc = {val_acc:.4f}")


mean_loss = sum(all_val_losses) / len(all_val_losses)
mean_acc = sum(all_val_accuracies) / len(all_val_accuracies)

print("\n📊 Cross-validation results:")
print(f"Average Validation Loss: {mean_loss:.4f}")
print(f"Average Validation Accuracy: {mean_acc:.4f}")
print(f"All Fold Accuracies: {all_val_accuracies}")

#All Fold Accuracies: [0.4812747150500116, 0.9596758633995756, 0.6296825222454135, 0.34934054785255325, 0.6497595102754701, 0.9419537517697026, 0.6900668576886342, 0.7086124401913876]

In [ ]:
# 📂 Fold 0: 4 validation groups
# Class distribution: {0: 465}
# Validation accuracy : 0.4812747150500116

# 📂 Fold 1: 4 validation groups
# Class distribution: {0: 185}
# Validation accuracy : 0.9596758633995756

# 📂 Fold 2: 4 validation groups
# Class distribution: {0: 16671, 1: 434, 2: 5545}
# Validation accuracy :  0.6296825222454135

# 📂 Fold 3: 4 validation groups
# Class distribution: {0: 267}
# Validation accuracy :  0.34934054785255325

# 📂 Fold 4: 3 validation groups
# Class distribution: {0: 206, 1: 145}
# Validation accuracy : 0.6497595102754701

# 📂 Fold 5: 3 validation groups
# Class distribution: {0: 4083, 2: 2, 1: 325}
# Validation accuracy : 0.9419537517697026

# 📂 Fold 6: 3 validation groups
# Class distribution: {0: 6526, 2: 6506}
# Validation accuracy : 0.6900668576886342

# 📂 Fold 7: 3 validation groups
# Class distribution: {0: 2553}
# Validation accuracy : 0.7086124401913876

# 📊 Cross-validation results:
# Average Validation Loss: 1.1046
# Average Validation Accuracy: 0.6763
# All Fold Accuracies: [0.4812747150500116, 0.9596758633995756, 0.6296825222454135, 0.34934054785255325, 0.6497595102754701, 0.9419537517697026, 0.6900668576886342, 0.7086124401913876]

In [ ]:
#@title Check alignment

def check_embeddings_alignment(embeddings_dict, dataset):
    """
    Checks whether the labels in the embeddings_dict match the labels
    reconstructed from the current dataset.entries order.
    Returns True if perfectly aligned, False otherwise.
    """
    labels_tensor = embeddings_dict["labels"].cpu().numpy()
    ids_tensor = embeddings_dict["ids"].cpu().numpy()

    mismatches = []
    for i, (label, idx) in enumerate(zip(labels_tensor, ids_tensor)):
        taxon, row = dataset.entries[int(idx)]
        true_label = dataset.label_map[row["label"]]
        if label != true_label:
            mismatches.append((i, idx, label, true_label, taxon))

    if not mismatches:
        print("✅ Embeddings are aligned with the current dataset order.")
        return True
    else:
        print(f"❌ Found {len(mismatches)} mismatches!")
        print("Example mismatches (up to 10):")
        for m in mismatches[:10000]:
            print(f"Embedding idx {m[0]} (obs_id={m[1]}): stored_label={m[2]}, true_label={m[3]}, taxon={m[4]}")
        return False

# Usage example:
embeddings_dict = torch.load('/content/drive/MyDrive/Thesis/bioclip_2_embeddings.pt')
dataset = InvasiveSpeciesDataset(root_dir='/content/ai4biological-pattern/barbara_new/taxas/')
check_embeddings_alignment(embeddings_dict, dataset)

### All species split (Leave One Group Out)

In [ ]:
all_val_losses = []
all_val_accuracies = []

n_splits = 28

final_HP_search = {'augmentation': True, 'loss_type': 'CE', 'factor': 2, 'val_loss': 0.35223526106626696, 'val_accuracy': 0.8417397244677217}
embeddings_save_path_BC2_new = '/content/drive/MyDrive/Thesis/NEW_GROUPS_bioclip_2_embeddings.pt'
model_name_final = 'group_k_fold_new_embeddings'
epochs=50

for fold_idx in range(n_splits):
  val_loss, val_acc = create_model_and_predict(folder_name="taxas", use_bioclip=True, augmentation=final_HP_search['augmentation'],
                                          augmentation_factor=final_HP_search['factor'], loss_type=final_HP_search['loss_type'],
                                         bioclip_version=2, output_dir="invasive_species", model_name=model_name_final,
                                           save_model=False, verbose=True, embeddings_save_path=embeddings_save_path_BC2_new,
                                             epochs=epochs, debug_subset_size=None, n_splits=n_splits, fold_idx=fold_idx)
  all_val_losses.append(val_loss)
  all_val_accuracies.append(val_acc)
  print(f"✅ Fold {fold_idx} done: Val Loss = {val_loss:.4f}, Val Acc = {val_acc:.4f}")


mean_loss = sum(all_val_losses) / len(all_val_losses)
mean_acc = sum(all_val_accuracies) / len(all_val_accuracies)

print("\n📊 Cross-validation results:")
print(f"Average Validation Loss: {mean_loss:.4f}")
print(f"Average Validation Accuracy: {mean_acc:.4f}")
print(f"All Fold Accuracies: {all_val_accuracies}")

In [ ]:
#JUST THE FINAL SPLITS

all_val_losses = []
all_val_accuracies = []

n_splits = 28

final_HP_search = {'augmentation': True, 'loss_type': 'CE', 'factor': 2, 'val_loss': 0.35223526106626696, 'val_accuracy': 0.8417397244677217}
embeddings_save_path_BC2_new = '/content/drive/MyDrive/Thesis/NEW_GROUPS_bioclip_2_embeddings.pt'
model_name_final = 'group_k_fold_new_embeddings'
epochs=50

for fold_idx in range(24,28):
  val_loss, val_acc = create_model_and_predict(folder_name="taxas", use_bioclip=True, augmentation=final_HP_search['augmentation'],
                                          augmentation_factor=final_HP_search['factor'], loss_type=final_HP_search['loss_type'],
                                         bioclip_version=2, output_dir="invasive_species", model_name=model_name_final,
                                           save_model=False, verbose=True, embeddings_save_path=embeddings_save_path_BC2_new,
                                             epochs=epochs, debug_subset_size=None, n_splits=n_splits, fold_idx=fold_idx)
  all_val_losses.append(val_loss)
  all_val_accuracies.append(val_acc)
  print(f"✅ Fold {fold_idx} done: Val Loss = {val_loss:.4f}, Val Acc = {val_acc:.4f}")


mean_loss = sum(all_val_losses) / len(all_val_losses)
mean_acc = sum(all_val_accuracies) / len(all_val_accuracies)

print("\n📊 Cross-validation results:")
print(f"Average Validation Loss: {mean_loss:.4f}")
print(f"Average Validation Accuracy: {mean_acc:.4f}")
print(f"All Fold Accuracies: {all_val_accuracies}")